# Simple MLP Model with MLflow 3.0+ Tracking

Demonstrates MLflow 3.0+ features during training with a simple Multi-Layer Perceptron.

**Architecture:**
- Input: Customer features + Article features (concatenated)
- Hidden layers: 256 → 128 → 64 units with ReLU activation
- Output: Binary classification (will customer buy this article?)

**MLflow 3.0+ Tracking:**
- Enable system metrics logging (CPU, GPU, memory, network, disk)
- Single run wrapping all training epochs
- Log model checkpoint per epoch (multiple models in one run)
- Log hyperparameters and per-epoch metrics
- Save final best model to Unity Catalog

**Expected Performance:** Baseline binary classification for purchase prediction

## Setup

In [ ]:
import sys

# Add project root to path (go up 2 levels from notebooks/)
sys.path.append("../../")

from pyspark.sql.functions import *
import mlflow
import mlflow.pytorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from config.catalog_config import get_table_config
from config.widget_utils import get_bundle_parameters
from utils.data_utils import load_delta_table

In [ ]:
# Get parameters from bundle (passed as notebook parameters)
params = get_bundle_parameters(model_default="simple_mlp_model")
catalog_name = params["catalog_name"]
schema_name = params["schema_name"]
experiment_name = params["experiment_name"]
model_name = params["model_name"]

print(f"Parameters:")
print(f"  Catalog: {catalog_name}")
print(f"  Schema: {schema_name}")
print(f"  Experiment: {experiment_name}")
print(f"  Model: {model_name}")

# Get table configuration
tables = get_table_config(catalog_name, schema_name)

## Configuration

In [ ]:
# Model configuration
CONFIG = {
    # Architecture
    "hidden_layers": [256, 128, 64],
    "dropout": 0.3,
    
    # Training
    "batch_size": 512,
    "num_epochs": 10,
    "learning_rate": 0.001,
    "weight_decay": 1e-5,
    
    # Data sampling
    "negative_sample_ratio": 2.0,  # Negative:Positive ratio
    "max_samples_per_epoch": 100000,  # Limit for faster training
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## Load Data

In [ ]:
print("Loading data from Delta tables...")

# Load transactions for creating positive/negative samples
train_df = load_delta_table(tables.TRAIN_SILVER)
val_df = load_delta_table(tables.VAL_SILVER)

# Load customer and article data for features
customers_df = load_delta_table(tables.CUSTOMERS_SOURCE)
articles_df = load_delta_table(tables.ARTICLES_SOURCE)

print(f"Train transactions: {train_df.count():,}")
print(f"Val transactions: {val_df.count():,}")
print(f"Customers: {customers_df.count():,}")
print(f"Articles: {articles_df.count():,}")

## Feature Engineering

In [ ]:
# Extract customer features
print("Extracting customer features...")

# Simple features: age encoded, active status, etc.
customers_features = customers_df.select(
    "customer_id",
    # Age encoding (handle nulls)
    when(col("age").isNull(), 0).otherwise(col("age")).alias("age"),
    # FN: Fashion News Frequency (0=None, 1=Monthly, 2=Regularly)
    when(col("FN") == 0, 0).otherwise(1).alias("is_subscribed"),
    when(col("Active").isNull(), 0).otherwise(col("Active")).alias("is_active"),
    # Club member status
    when(col("club_member_status").isNull(), 0)
        .when(col("club_member_status") == "ACTIVE", 1)
        .otherwise(0).alias("is_club_member"),
)

print(f"Customer features extracted: {customers_features.count():,} rows")
customers_features.show(5)

In [ ]:
# Extract article features
print("Extracting article features...")

from pyspark.ml.feature import StringIndexer
from pyspark.sql.window import Window

# Encode categorical features
articles_features = articles_df.select(
    "article_id",
    # Product type encoding (simplified)
    when(col("product_type_no").isNull(), 0).otherwise(col("product_type_no")).alias("product_type"),
    # Product group encoding
    when(col("product_group_name") == "Garment Upper body", 1)
        .when(col("product_group_name") == "Garment Lower body", 2)
        .when(col("product_group_name") == "Shoes", 3)
        .when(col("product_group_name") == "Accessories", 4)
        .otherwise(0).alias("product_group_encoded"),
    # Garment group encoding (simplified to first 3 digits)
    when(col("garment_group_no").isNull(), 0)
        .otherwise((col("garment_group_no") / 1000).cast("int")).alias("garment_group"),
    # Color encoding (simplified)
    when(col("colour_group_code").isNull(), 0).otherwise(col("colour_group_code")).alias("colour_code"),
)

print(f"Article features extracted: {articles_features.count():,} rows")
articles_features.show(5)

## Create Training Dataset

Create positive samples (actual purchases) and negative samples (random non-purchases)

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number

def create_binary_dataset(transactions_df, customers_features, articles_features, 
                          negative_ratio=2.0, max_samples=100000):
    """
    Create binary classification dataset:
    - Positive samples: actual purchases (label=1)
    - Negative samples: random customer-article pairs (label=0)
    """
    print(f"Creating binary dataset with negative ratio={negative_ratio}...")
    
    # Positive samples (actual purchases)
    positive_samples = transactions_df.select(
        "customer_id",
        "article_id"
    ).distinct().withColumn("label", lit(1))
    
    num_positive = positive_samples.count()
    print(f"Positive samples: {num_positive:,}")
    
    # Sample positive examples if too many
    if num_positive > max_samples:
        fraction = max_samples / num_positive
        positive_samples = positive_samples.sample(fraction=fraction, seed=42)
        num_positive = positive_samples.count()
        print(f"Sampled positive: {num_positive:,}")
    
    # Negative samples (random customer-article pairs that weren't purchased)
    num_negative = int(num_positive * negative_ratio)
    print(f"Generating {num_negative:,} negative samples...")
    
    # Get sample of customers and articles
    customers_sample = customers_features.select("customer_id").sample(fraction=0.3, seed=42)
    articles_sample = articles_features.select("article_id").sample(fraction=0.3, seed=43)
    
    # Cross join to create random pairs
    random_pairs = customers_sample.crossJoin(articles_sample)
    
    # Remove pairs that exist in positive samples (actual purchases)
    negative_samples = random_pairs.join(
        positive_samples.select("customer_id", "article_id"),
        on=["customer_id", "article_id"],
        how="left_anti"
    ).withColumn("label", lit(0))
    
    # Sample to get desired number
    negative_samples = negative_samples.limit(num_negative)
    
    print(f"Negative samples created: {negative_samples.count():,}")
    
    # Combine positive and negative
    combined = positive_samples.union(negative_samples)
    
    # Join with features
    dataset = combined \
        .join(customers_features, on="customer_id", how="inner") \
        .join(articles_features, on="article_id", how="inner")
    
    print(f"Total dataset size: {dataset.count():,}")
    
    return dataset

# Create training dataset
train_dataset = create_binary_dataset(
    train_df,
    customers_features,
    articles_features,
    negative_ratio=CONFIG["negative_sample_ratio"],
    max_samples=CONFIG["max_samples_per_epoch"]
)

print("\nTrain dataset sample:")
train_dataset.show(5)

In [ ]:
# Create validation dataset
val_dataset = create_binary_dataset(
    val_df,
    customers_features,
    articles_features,
    negative_ratio=CONFIG["negative_sample_ratio"],
    max_samples=CONFIG["max_samples_per_epoch"] // 2  # Smaller val set
)

## Prepare PyTorch Datasets

In [ ]:
# Convert to pandas and prepare features
print("Converting to pandas...")

# Select feature columns
feature_cols = ["age", "is_subscribed", "is_active", "is_club_member",
                "product_type", "product_group_encoded", "garment_group", "colour_code"]

train_pd = train_dataset.select(*feature_cols, "label").toPandas()
val_pd = val_dataset.select(*feature_cols, "label").toPandas()

print(f"Train samples: {len(train_pd):,}")
print(f"Val samples: {len(val_pd):,}")

# Check class balance
train_pos_ratio = train_pd["label"].mean()
val_pos_ratio = val_pd["label"].mean()
print(f"\nTrain positive ratio: {train_pos_ratio:.3f}")
print(f"Val positive ratio: {val_pos_ratio:.3f}")

In [ ]:
# Normalize features
print("Normalizing features...")

scaler = StandardScaler()

X_train = train_pd[feature_cols].values
y_train = train_pd["label"].values

X_val = val_pd[feature_cols].values
y_val = val_pd["label"].values

# Fit scaler on training data
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print(f"Feature dimension: {X_train_scaled.shape[1]}")
print(f"Train shape: {X_train_scaled.shape}")
print(f"Val shape: {X_val_scaled.shape}")

In [ ]:
# Create PyTorch datasets
class SimpleDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset_pt = SimpleDataset(X_train_scaled, y_train)
val_dataset_pt = SimpleDataset(X_val_scaled, y_val)

# Create dataloaders
train_loader = DataLoader(
    train_dataset_pt,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset_pt,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=0
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## Define Simple MLP Model

In [ ]:
class SimpleMLP(nn.Module):
    """
    Simple Multi-Layer Perceptron for binary classification
    """
    def __init__(self, input_dim, hidden_layers=[256, 128, 64], dropout=0.3):
        super(SimpleMLP, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_layers = hidden_layers
        
        # Build layers
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_layers:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, 1))
        layers.append(nn.Sigmoid())
        
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x).squeeze()

# Create model
input_dim = X_train_scaled.shape[1]
model = SimpleMLP(
    input_dim=input_dim,
    hidden_layers=CONFIG["hidden_layers"],
    dropout=CONFIG["dropout"]
)

print(f"Model created:")
print(f"  Input dim: {input_dim}")
print(f"  Hidden layers: {CONFIG['hidden_layers']}")

print(f"\nModel architecture:")
print(model)

## Training with MLflow Tracking

In [ ]:
# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Using device: {device}")

# Loss and optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"]
)

In [ ]:
# Initialize MLflow
mlflow.set_experiment(experiment_name)

# ========== PREPARE MLFLOW DATASETS (MLflow 3.0+) ==========
print("\n" + "="*60)
print("PREPARING MLFLOW DATASETS")
print("="*60)

# Create MLflow datasets for tracking
train_dataset = mlflow.data.from_pandas(train_pd, name="train")
val_dataset = mlflow.data.from_pandas(val_pd, name="validation")

print("✓ Train and validation datasets created for MLflow tracking")
print(f"  Train dataset: {len(train_pd):,} samples")
print(f"  Val dataset: {len(val_pd):,} samples")

# ========== ENABLE SYSTEM METRICS LOGGING (MLflow 3.0+) ==========
print("\n" + "="*60)
print("ENABLING MLFLOW 3.0+ SYSTEM METRICS LOGGING")
print("="*60)

# Start MLflow run with system metrics - wraps entire training with all epochs
with mlflow.start_run(run_name="simple_mlp_training", log_system_metrics=True) as run:
    
    run_id = run.info.run_id
    print("✓ System metrics logging enabled (CPU, GPU, memory, network, disk)")
    print("  Metrics will be automatically logged during training")
    
    # ========== LOG HYPERPARAMETERS ==========
    print("\n" + "="*60)
    print("LOGGING HYPERPARAMETERS TO MLFLOW")
    print("="*60)
    
    mlflow.log_param("model_type", "simple_mlp")
    mlflow.log_param("input_dim", input_dim)
    mlflow.log_param("hidden_layers", str(CONFIG["hidden_layers"]))
    mlflow.log_param("dropout", CONFIG["dropout"])
    mlflow.log_param("batch_size", CONFIG["batch_size"])
    mlflow.log_param("num_epochs", CONFIG["num_epochs"])
    mlflow.log_param("learning_rate", CONFIG["learning_rate"])
    mlflow.log_param("weight_decay", CONFIG["weight_decay"])
    mlflow.log_param("negative_sample_ratio", CONFIG["negative_sample_ratio"])
    
    print("✓ Hyperparameters logged to MLflow")
    
    # ========== TRAINING LOOP ==========
    print("\n" + "="*60)
    print("STARTING TRAINING WITH PER-EPOCH MLFLOW LOGGING")
    print("="*60)
    
    best_val_loss = float('inf')
    best_val_accuracy = 0.0
    best_epoch = 0
    history = {
        'train_loss': [],
        'train_accuracy': [],
        'val_loss': [],
        'val_accuracy': []
    }
    
    for epoch in range(CONFIG["num_epochs"]):
        print(f"\nEpoch {epoch+1}/{CONFIG['num_epochs']}")
        print("-" * 40)
        
        # ========== TRAINING PHASE ==========
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            # Forward pass
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            # Metrics
            train_loss += loss.item()
            predicted = (outputs > 0.5).float()
            train_correct += (predicted == y_batch).sum().item()
            train_total += y_batch.size(0)
            
            # Print progress every 20 batches
            if (batch_idx + 1) % 20 == 0:
                print(f"  Batch {batch_idx+1}/{len(train_loader)} - Loss: {loss.item():.4f}")
        
        # Calculate epoch metrics
        avg_train_loss = train_loss / len(train_loader)
        train_accuracy = train_correct / train_total
        
        history['train_loss'].append(avg_train_loss)
        history['train_accuracy'].append(train_accuracy)
        
        # ========== VALIDATION PHASE ==========
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                
                val_loss += loss.item()
                predicted = (outputs > 0.5).float()
                val_correct += (predicted == y_batch).sum().item()
                val_total += y_batch.size(0)
        
        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = val_correct / val_total
        
        history['val_loss'].append(avg_val_loss)
        history['val_accuracy'].append(val_accuracy)
        
        print(f"  Train Loss: {avg_train_loss:.4f} | Train Acc: {train_accuracy:.4f}")
        print(f"  Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.4f}")
        
        # ========== LOG MODEL CHECKPOINT (MLflow 3.0+) ==========
        print(f"  💾 Logging model checkpoint for epoch {epoch+1}...")
        
        # Get sample input and output for signature
        sample_batch = next(iter(train_loader))[0][:5].cpu()
        sample_input = sample_batch.numpy()
        
        # Generate sample output for signature
        model.eval()
        with torch.no_grad():
            sample_output = model(sample_batch.to(device)).cpu().numpy()
        model.train()
        
        # Infer signature (required for Unity Catalog)
        signature = mlflow.models.infer_signature(sample_input, sample_output)
        
        # Log model checkpoint with artifact_path, name, params, step, signature, and input_example
        model_info = mlflow.pytorch.log_model(
            pytorch_model=model,
            name=f"simple-mlp-epoch-{epoch+1}",
            params={
                "architecture": "MLP",
                "n_layers": len(CONFIG["hidden_layers"]) + 2,
                "activation": "ReLU",
                "criterion": "BCELoss",
                "optimizer": "Adam",
                "dropout": CONFIG["dropout"],
                "batch_size": CONFIG["batch_size"],
                "learning_rate": CONFIG["learning_rate"],
            },
            step=epoch,
            signature=signature,
            input_example=sample_input,
        )
        
        # ========== LOG METRICS LINKED TO MODEL AND DATASET (MLflow 3.0+) ==========
        # Link metrics to specific LoggedModel and dataset
        mlflow.log_metric(
            key="train/loss",
            value=avg_train_loss,
            step=epoch,
            model_id=model_info.model_id,
            dataset=train_dataset,
        )
        mlflow.log_metric(
            key="train/accuracy",
            value=train_accuracy,
            step=epoch,
            model_id=model_info.model_id,
            dataset=train_dataset,
        )
        mlflow.log_metric(
            key="val/loss",
            value=avg_val_loss,
            step=epoch,
            model_id=model_info.model_id,
            dataset=val_dataset,
        )
        mlflow.log_metric(
            key="val/accuracy",
            value=val_accuracy,
            step=epoch,
            model_id=model_info.model_id,
            dataset=val_dataset,
        )
        
        print(f"  ✓ Checkpoint saved with model_id: {model_info.model_id}")
        
        # Save best model state
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_val_accuracy = val_accuracy
            best_epoch = epoch + 1
            best_model_state = model.state_dict().copy()
            best_model_id = model_info.model_id
            print(f"  ⭐ New best model! (epoch {best_epoch}, val_loss: {best_val_loss:.4f})")
    
    print("\n" + "="*60)
    print("TRAINING COMPLETE")
    print("="*60)
    print(f"Best validation loss: {best_val_loss:.4f} at epoch {best_epoch}")
    
    # Load best model
    model.load_state_dict(best_model_state)
    
    # ========== LOG FINAL METRICS ==========
    mlflow.log_metric("best_val_loss", best_val_loss)
    mlflow.log_metric("best_val_accuracy", best_val_accuracy)
    mlflow.log_metric("best_epoch", best_epoch)
    mlflow.log_metric("final_train_accuracy", history['train_accuracy'][-1])
    mlflow.log_metric("final_val_accuracy", history['val_accuracy'][-1])
    
    # ========== SAVE FINAL BEST MODEL TO MLFLOW ==========
    print("\n📦 Saving final best model to MLflow...")
    
    # Get sample for signature inference
    sample_batch = next(iter(train_loader))[0][:5].cpu()
    sample_input = sample_batch.numpy()
    
    # Generate output for signature
    model.eval()
    with torch.no_grad():
        sample_output = model(sample_batch.to(device)).cpu().numpy()
    
    # Infer signature (required for Unity Catalog registration)
    signature = mlflow.models.infer_signature(sample_input, sample_output)
    
    # Save best model artifact with signature
    final_model_info = mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path="model",
        signature=signature,
        input_example=sample_input,
    )
    
    # Save scaler
    import joblib
    scaler_path = "/tmp/scaler.pkl"
    joblib.dump(scaler, scaler_path)
    mlflow.log_artifact(scaler_path, "preprocessor")
    
    # Save feature columns info
    mlflow.log_dict({"feature_columns": feature_cols}, "feature_columns.json")
    
    # Log tags
    mlflow.set_tag("stage", "training")
    mlflow.set_tag("model_type", "simple_mlp")
    mlflow.set_tag("framework", "pytorch")
    mlflow.set_tag("mlflow_version", "3.0+")
    mlflow.set_tag("features", "system_metrics,logged_models,dataset_linking")
    mlflow.set_tag("best_model_id", best_model_id)
    
    print("✓ Final best model and artifacts saved to MLflow")
    print(f"✓ Model signature: {signature}")
    
    print(f"\n✓ MLflow Run ID: {run_id}")
    print(f"\n📊 MLflow 3.0+ Features Used:")
    print(f"  ✓ System metrics logging (log_system_metrics=True)")
    print(f"  ✓ Multiple LoggedModels ({CONFIG['num_epochs']} epochs)")
    print(f"  ✓ Model signatures for Unity Catalog compatibility")
    print(f"  ✓ Metrics linked to model_id and dataset")
    print(f"  ✓ Single run wrapping all epochs")
    print(f"  ✓ Best model: epoch {best_epoch} (model_id: {best_model_id})")

## Search and Rank Checkpoints (MLflow 3.0+)

MLflow 3.0+ allows searching and ranking LoggedModels by performance metrics.

In [ ]:
from mlflow import MlflowClient

print("\n" + "="*60)
print("SEARCHING AND RANKING LOGGED MODELS")
print("="*60)

client = MlflowClient()

# 1. Get run information and all logged models
run = client.get_run(run_id)
model_outputs = run.outputs.model_outputs

print(f"\n✓ Found {len(model_outputs)} LoggedModels in this run")

# 2. Collect detailed information for each model
model_info_list = []

for model_output in model_outputs:
    model_id = model_output.model_id
    step = model_output.step
    
    try:
        # Get model information by model_id
        model_info = client.get_logged_model(model_id)
        
        # Get metrics for this step
        try:
            # Get val/accuracy metric history
            metric_history = client.get_metric_history(run_id, "val/accuracy")
            
            # Find metric for this step
            val_accuracy = None
            for metric in metric_history:
                if metric.step == step:
                    val_accuracy = metric.value
                    break
            
            # If no exact match, find closest step
            if val_accuracy is None and len(metric_history) > 0:
                closest_metric = min(metric_history, key=lambda m: abs(m.step - step))
                val_accuracy = closest_metric.value
                
            # Get val/loss
            loss_history = client.get_metric_history(run_id, "val/loss")
            val_loss = None
            for metric in loss_history:
                if metric.step == step:
                    val_loss = metric.value
                    break
        
        except Exception as e:
            print(f"  Failed to get metrics for step {step}: {e}")
            val_accuracy = 0
            val_loss = 0
        
        model_info_list.append({
            'model_id': model_id,
            'step': step,
            'epoch': step + 1,  # epochs are 1-indexed
            'val_accuracy': val_accuracy if val_accuracy is not None else 0,
            'val_loss': val_loss if val_loss is not None else 0,
            'model_uri': model_info.model_uri if hasattr(model_info, 'model_uri') else f"models:/{model_id}"
        })
        
    except Exception as e:
        print(f"  Failed to get info for model {model_id} (step {step}): {e}")

# 3. Rank by validation accuracy (descending)
ranked_models = sorted(
    model_info_list,
    key=lambda x: x['val_accuracy'],
    reverse=True
)

print(f"\n{'='*70}")
print(f"CHECKPOINT RANKING BY VALIDATION ACCURACY")
print(f"{'='*70}")
print(f"Total checkpoints: {len(ranked_models)}")

# 4. Display top 3 models
import pandas as pd

if len(ranked_models) >= 3:
    top3_df = pd.DataFrame([
        {
            "Rank": i + 1,
            "Epoch": model['epoch'],
            "Val Accuracy": f"{model['val_accuracy']:.4f}",
            "Val Loss": f"{model['val_loss']:.4f}",
            "Model ID": model['model_id'][:16] + "..."
        }
        for i, model in enumerate(ranked_models[:3])
    ])
    
    print("\nTop 3 Models:")
    print(top3_df.to_string(index=False))
else:
    print("\nAll Models:")
    for i, model in enumerate(ranked_models):
        print(f"{i+1}. Epoch {model['epoch']}: Val Acc={model['val_accuracy']:.4f}, Val Loss={model['val_loss']:.4f}")

# 5. Best model information
if ranked_models:
    best_model = ranked_models[0]
    print(f"\n{'='*70}")
    print(f"BEST PERFORMING MODEL")
    print(f"{'='*70}")
    print(f"Model ID: {best_model['model_id']}")
    print(f"Epoch: {best_model['epoch']}")
    print(f"Val Accuracy: {best_model['val_accuracy']:.4f}")
    print(f"Val Loss: {best_model['val_loss']:.4f}")
    print(f"Model URI: {best_model['model_uri']}")
    
    print(f"\n💡 Load this model:")
    print(f"   # Using model URI")
    print(f"   model = mlflow.pytorch.load_model('{best_model['model_uri']}')")
    print(f"\n   # Using model ID")
    print(f"   model = mlflow.pytorch.load_model('models:/{best_model['model_id']}')")

print("\n" + "="*70)

## Training Curves Visualization

In [ ]:
import matplotlib.pyplot as plt

# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

epochs_range = range(1, CONFIG["num_epochs"] + 1)

# Loss plot
ax1.plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
ax1.plot(epochs_range, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(epochs_range, history['train_accuracy'], 'b-', label='Train Accuracy', linewidth=2)
ax2.plot(epochs_range, history['val_accuracy'], 'r-', label='Val Accuracy', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
display(fig)

# Log figure to MLflow
with mlflow.start_run(run_id=run_id):
    mlflow.log_figure(fig, "training_curves.png")
    print("✓ Training curves logged to MLflow")

## Register Best Model to Unity Catalog

In [ ]:
print("Registering model to Unity Catalog Model Registry...")

# Register to Unity Catalog
uc_model_name = f"{catalog_name}.{schema_name}.{model_name}"
model_uri = f"runs:/{run_id}/model"

print(f"\nRegistering model to Unity Catalog: {uc_model_name}")

registered_model = mlflow.register_model(model_uri, uc_model_name)

print(f"✓ Model registered: {uc_model_name}")
print(f"✓ Version: {registered_model.version}")
print(f"✓ Run ID: {run_id}")

# Add model description
from mlflow.tracking import MlflowClient

client = MlflowClient()
client.update_registered_model(
    name=uc_model_name,
    description=f"Simple MLP for binary classification (purchase prediction). "
    f"Best Val Accuracy: {best_val_accuracy:.4f} at epoch {best_epoch}.",
)

client.update_model_version(
    name=uc_model_name,
    version=registered_model.version,
    description=f"MLP architecture: {input_dim} → {' → '.join(map(str, CONFIG['hidden_layers']))} → 1. "
    f"Best Val Loss: {best_val_loss:.4f}, Best Val Acc: {best_val_accuracy:.4f} (epoch {best_epoch}). "
    f"Trained for {CONFIG['num_epochs']} epochs with batch size {CONFIG['batch_size']}. "
    f"Model ID: {best_model_id}",
)

print(f"\n✓ Model {uc_model_name} v{registered_model.version} registered successfully!")

## Summary

In [ ]:
print("\n" + "="*60)
print("SIMPLE MLP TRAINING COMPLETE")
print("="*60)
print(f"Model Type: Simple Multi-Layer Perceptron")
print(f"Task: Binary Classification (Purchase Prediction)")
print(f"")
print(f"Architecture:")
print(f"  - Input: {input_dim} features")
print(f"  - Hidden: {' → '.join(map(str, CONFIG['hidden_layers']))} units")
print(f"  - Output: 1 (sigmoid activation)")
print(f"")
print(f"Training Data:")
print(f"  - Train samples: {len(train_pd):,}")
print(f"  - Val samples: {len(val_pd):,}")
print(f"  - Negative sample ratio: {CONFIG['negative_sample_ratio']}")
print(f"")
print(f"Performance:")
print(f"  - Best Val Loss: {best_val_loss:.4f} (epoch {best_epoch})")
print(f"  - Best Val Accuracy: {best_val_accuracy:.4f}")
print(f"  - Final Train Accuracy: {history['train_accuracy'][-1]:.4f}")
print(f"  - Final Val Accuracy: {history['val_accuracy'][-1]:.4f}")
print(f"")
print(f"MLflow Tracking:")
print(f"  - Experiment: {experiment_name}")
print(f"  - Run ID: {run_id}")
print(f"  - Model: {uc_model_name}")
print(f"  - Version: {registered_model.version}")
print(f"  - Best Model ID: {best_model_id}")
print(f"")
print(f"MLflow 3.0+ Features:")
print(f"  ✓ System metrics logged automatically")
print(f"  ✓ {CONFIG['num_epochs']} LoggedModels created")
print(f"  ✓ Metrics linked to model_id and dataset")
print(f"  ✓ Single run for entire training session")
print(f"  ✓ Model ranking by validation accuracy")
print("="*60)
print("\n✓ Training complete with full MLflow 3.0+ tracking!")
print("\n💡 Key MLflow 3.0+ Features Demonstrated:")
print("  1. log_system_metrics=True - Automatic system metrics logging")
print("  2. mlflow.data.from_pandas() - Dataset tracking")
print("  3. mlflow.pytorch.log_model() - LoggedModel with name, params, step")
print("  4. mlflow.log_metric() - Metrics linked to model_id and dataset")
print("  5. run.outputs.model_outputs - Access all LoggedModels")
print("  6. client.get_logged_model() - Query LoggedModel by ID")
print("  7. Model ranking and comparison by performance")
print("  8. Unity Catalog model registration")
print("="*60)

print("\n💡 Understanding MLflow 3.0+ Concepts:")
print("  - Run: One training job (this entire notebook execution)")
print(f"  - LoggedModels: {CONFIG['num_epochs']} checkpoint models within the run")
print("  - model_id: Unique identifier for each LoggedModel")
print("  - Dataset: Train/val data tracked with metrics")
print("  - Metrics: Linked to specific LoggedModel and dataset")
print("\n💡 Next Steps:")
print("  1. View system metrics in MLflow UI → 'System Metrics' tab")
print("  2. Compare LoggedModels in MLflow UI → 'Models' tab")
print("  3. Use best model for inference or deployment")
print("  4. Load any checkpoint: mlflow.pytorch.load_model(model_uri)")
print("="*60)